# 🌸 Lily LTX 2B Distilled — Fast Image → Video

Run this one cell with **Internet ON** and a Kaggle **GPU** attached.

Pipeline: **image → LTX 2B distilled → short low-res native clip → cheap upscale + FPS conversion → MP4**.


In [ ]:
import urllib.request, pathlib, shutil, zipfile
ROOT = pathlib.Path('/kaggle/working')
LTX_DIR = ROOT / 'LTX-Video'
shutil.rmtree(LTX_DIR, ignore_errors=True)
shutil.rmtree(ROOT / 'LTX-Video-src', ignore_errors=True)
PIN = '4b2d053057623ddd4d0a1d3e9cd28890e9ef487f'
ZIP = ROOT / 'ltx_source.zip'
print('⬇️ Downloading pinned LTX source ZIP (no git)...')
urllib.request.urlretrieve(f'https://github.com/Lightricks/LTX-Video/archive/{PIN}.zip', ZIP)
with zipfile.ZipFile(ZIP, 'r') as z:
    z.extractall(ROOT)
extracted = ROOT / f'LTX-Video-{PIN}'
extracted.rename(LTX_DIR)
ZIP.unlink(missing_ok=True)
print('✅ Pinned LTX source ready:', LTX_DIR)
URL = 'https://raw.githubusercontent.com/benruiz1024-ops/hi/main/LILY_LTX2B_FAST_I2V_STUDIO.py'
DST = ROOT / 'LILY_LTX2B_FAST_I2V_STUDIO.py'
urllib.request.urlretrieve(URL, DST)
code = DST.read_text(encoding='utf-8')
# Xet support for current Hugging Face large-file storage.
code = code.replace('\"huggingface-hub==0.30.2\",', '\"huggingface-hub==0.30.2\",\n    \"hf-xet>=1.1.5\",')
# Replace all git-based LTX setup with the already-extracted pinned source tree.
old_git = '''if not LTX_REPO.exists():
    run(["git", "clone", "https://github.com/Lightricks/LTX-Video.git", str(LTX_REPO)])
run(["git", "fetch", "origin", PIN], cwd=LTX_REPO)
run(["git", "checkout", "--force", PIN], cwd=LTX_REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(LTX_REPO), "--no-deps"])'''
new_git = '''if not LTX_REPO.exists():
    raise RuntimeError(f"Pinned LTX source folder is missing: {LTX_REPO}")
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(LTX_REPO), "--no-deps"])'''
if old_git not in code:
    raise RuntimeError('Launcher patch could not find the git setup block; refusing to run stale code.')
code = code.replace(old_git, new_git)
# Avoid the brittle PixArt subfolder resolver entirely: snapshot exact files locally first.
code = code.replace('from huggingface_hub import hf_hub_download', 'from huggingface_hub import hf_hub_download, snapshot_download')
old_t5 = '''print("📝 Loading PixArt T5 text encoder in CPU RAM (FP16 storage)...")
TOKENIZER = T5Tokenizer.from_pretrained(TEXT_REPO, subfolder="tokenizer")
TEXT_ENCODER = T5EncoderModel.from_pretrained(
    TEXT_REPO,
    subfolder="text_encoder",
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
).cpu().eval()'''
new_t5 = '''print("📝 Downloading T5/tokenizer files explicitly (big first-run download)...")
TEXT_SNAPSHOT = snapshot_download(
    repo_id=TEXT_REPO,
    allow_patterns=["tokenizer/*", "text_encoder/*"],
    local_dir="/kaggle/working/pixart_text",
)
TOKENIZER = T5Tokenizer.from_pretrained(str(Path(TEXT_SNAPSHOT) / "tokenizer"), local_files_only=True)
TEXT_ENCODER = T5EncoderModel.from_pretrained(
    str(Path(TEXT_SNAPSHOT) / "text_encoder"),
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    local_files_only=True,
).cpu().eval()'''
if old_t5 not in code:
    raise RuntimeError('Launcher patch could not find the T5 loading block; refusing to run stale code.')
code = code.replace(old_t5, new_t5)
DST.write_text(code, encoding='utf-8')
print('✅ Studio patched: NO GIT + local T5 snapshot:', DST)
exec(compile(code, str(DST), 'exec'))
